# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 04 P5 clean benchmark analysis

**Permitted partitions by stage:** `smoke`: TRAIN ONLY (engineering check); `select`: TRAIN + VALIDATION; `test`: FINAL TEST — READ ONLY (plus historical inputs). Test access requires a completed immutable validation selection. No stage silently tunes on test data.

Start with smoke in a fresh CPU Colab runtime. Configure `AIRSENSE_GITHUB_TOKEN` and persistent model storage (for example a dedicated Google Drive directory). Compact artifacts push to a run branch; large native model files stay in persistent storage. A full selection run has 54 tuning fits and 9 learned refits, and can take hours on CPU. Do not use smoke outputs as research results. See `docs/P5_WORKFLOW.md`.

## 1. Bootstrap
Use the code SHA from the same GitHub revision as this notebook. Installing the declared environment can require a kernel restart; preserve the checkout and rerun setup.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = (os.environ.get("AIRSENSE_CODE_REVISION") or input("Code version: full 40-character Git commit SHA (not a run ID): ")).strip().lower()
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p5.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)


## 2. Provenance and GitHub authentication
Use a new run ID for new code. Reuse the benchmark ID for select, test and recovery; use a separate smoke ID.

In [ ]:
import logging
from airsense_r.artifacts import provenance, json_bytes
from airsense_r.artifacts import source_hash
from airsense_r.checkpoints import open_run
from airsense_r.colab import configure_github_secret

logging.basicConfig(level=logging.INFO)
if source_hash(ROOT) != "fec10ed8f2a06c91ed372d54799aa04aecd7143dadcc9c28c997ff59b1f486f3":
    raise RuntimeError("Notebook and scientific source revisions differ; open the matching notebook revision")
if TEST_MODE:
    RUN_ID = os.environ["AIRSENSE_NOTEBOOK_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_REMOTE")
else:
    from google.colab import userdata
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    RUN_ID = input("Run name, e.g. p5-smoke-01 (same name only for resume; not the code SHA): ").strip()
    REMOTE = REPOSITORY


## 3. Stage and persistent storage
Choose smoke first. After it passes, rerun this notebook with a new benchmark run ID and stage select. Later use that same benchmark ID with stage test. Use the same persistent directory across these stages. Start only one writer per run.

In [ ]:
STAGE = input("Stage [smoke/select/test] (default smoke): ").strip() or "smoke"
if STAGE not in {"smoke", "select", "test"}:
    raise ValueError("Choose smoke, select, or test")
storage_text = input("Persistent model directory (e.g. /content/drive/MyDrive/AirSense-R/model-checkpoints): ").strip()
if not storage_text:
    raise ValueError("A persistent model directory is required")
MODEL_DIRECTORY = Path(storage_text).expanduser().resolve()
if str(MODEL_DIRECTORY).startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted")
elif str(MODEL_DIRECTORY).startswith("/content/"):
    raise ValueError("Ordinary /content storage is ephemeral; use mounted persistent storage")
print("Stage:", STAGE)
print("Model storage:", MODEL_DIRECTORY)


## 4. Execute the reproducible script
Progress is checkpointed each neural epoch or tree-round chunk. A lost in-flight unit is repeated. GitHub failures stop the run; rerun the same identity to retry. A Drive mount confirms filesystem access, not a guarantee of immediate cloud-server flush.

In [ ]:
command = [sys.executable, str(ROOT / "scripts/run_benchmark.py"), "--stage", STAGE,
           "--run-id", RUN_ID, "--model-directory", str(MODEL_DIRECTORY), "--sync-remote", REMOTE]
subprocess.run(command, cwd=ROOT, check=True)


## 5. Inspect verified evidence
Engineering smoke reports contain recovery checks, not benchmark scores. For benchmarks, means are computed over seed metrics within station, then stations equally.

In [ ]:
import json
import pandas as pd
from airsense_r.artifacts import RunStore
output = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((output / "manifest.json").read_text())
store = RunStore(output, manifest["provenance"])
store.verify()
if STAGE == "smoke":
    display(pd.DataFrame([store.load_json(f"smoke-{family}.json") for family in ("xgboost", "mlp", "gru")]))
else:
    from airsense_r.p5 import require_selection
    display(require_selection(store))
    phase = "test" if STAGE == "test" else "validation"
    display(pd.read_csv(output / f"metrics-{phase}-aggregate.csv"))
    from IPython.display import Image
    display(Image(filename=str(output / f"figure-{phase}-mae.png")))
    if STAGE == "test":
        display(store.load_json("test-uncertainty.json"))


## Interpretation and next steps
A numerical rank change is not automatically meaningful. Interpret errors, the frozen near-tie threshold, paired intervals and selection regret together. Final-test winners are descriptive oracles only. P6 reuses the fitted models without retraining; retain their persistent snapshots. A changed configuration/environment requires a new run ID. Do not delete active result branches or model storage. Historical runs remain tied to their original code.